In [1]:
import requests
from bs4 import BeautifulSoup

import asyncio
import aiohttp
from itertools import chain

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

async def fetch_url(session, url):
    async with session.get(url, headers=headers) as response:
        return await response.text()

async def fetch_all_urls():
    urls = [f"https://nl.go.kr/NL/search/openApi/saseoApi.do?key=e2f8a1f4b65de08095f911a80ef8e144c81a27af807c134390bfcf036ed9554c&startRowNumApi=1&endRowNumApi=400&start_date=20200101&end_date=20250601&drcode={i}" for i in [11, 6, 5, 4]]
    
    async with aiohttp.ClientSession() as session:
        tasks = [fetch_url(session, url) for url in urls]
        responses = await asyncio.gather(*tasks)
        
    return responses

# Run async code in Jupyter
import nest_asyncio
nest_asyncio.apply()

responses = asyncio.run(fetch_all_urls())
all_soups = [BeautifulSoup(resp, 'xml') for resp in responses]

In [2]:
# Combine all soups into one list
combined_items = []
for soup in all_soups:
    items = soup.find_all('item')
    combined_items.extend(items)

len(combined_items)

1364

In [3]:
data = combined_items

In [5]:
json_data = []

import requests
APIKey = "e048f12e7d252e022a1ef06717d9d4b9"
base_url = "https://dapi.kakao.com/v3/search/book?target=isbn"
headers = {"Authorization": f"KakaoAK {APIKey}"}

for d in combined_items:
    title = d.find("recomtitle").text
    author = d.find("recomauthor").text
    publisher = d.find("recompublisher").text
    isbn = d.find("recomisbn").text.split(" ")[-1]
    content = BeautifulSoup(d.find("recomcontens").string, 'html.parser').text.replace("\n", "").strip()

    params = {
        "query": str(isbn),
        "sort": "accuracy",
        "page": 1,
        "size": 10,
        "target": "ISBN",
    }

    print(title, isbn)

    response = requests.get(base_url, headers=headers, params=params)
    result = response.json()

    try:
        book_result = result["documents"][0]

        book_content = BeautifulSoup(book_result["contents"], 'html.parser').text.replace("\n", "").strip()

        json_data.append(
        {
            "title": title,
            "author": author,
            "publisher": publisher,
            "isbn": isbn,
            "content": book_content,
            "recommendation": content,
        })
    except IndexError:
        print(f"No book found for ISBN: {isbn}, {title}")

 노화는 인간이 정복해야 할 마지막 병일까? 젊음은 모두에게 행복일까? 
 『텔로미어』는 이러한 도발적인 질문을 통해 생명 진화의 근본 의미를 탐구한다. 작품의 제목인 텔로미어는 염색체 끝에 위치한 구조로, 세포 분열시 DNA가 손상되지 않도록 보호하는 역할을 하며 노화와 수명을 결정짓는 핵심 요소다. 작가는 이러한 과학적 사실을 바탕으로 인위적으로 영생이 가능해진 미래의 세상을 창조해내고, 생명의 비밀에 대한 철학적 질문으로 우리를 초대한다. 
 젊음을 찾아주는 신약 ‘텔로프록산’을 75세 이상의 건강한 노인에게 의무적으로 투여하는 ‘노화종말법’의 시행을 앞둔 어느날, 기이한 살인사건이 연이어 발생한다. 고령의 치매 환자 어머니를 돌보는 형사 현묵은 이 사건을 수사하며 점차 불편한 진실과 마주한다. 현묵이 사건의 실마리를 풀어가는 과정은 ‘건강한 노인’만을 선별하는 이 제도가 만들어낸 새로운 차별과 소외, 인류의 불멸에 대한 욕망, 그리고 과학이 지닌 양면성을 날카롭게 드러낸다. 

과학 미스터리를 즐기는 독자와 초고령 사회의 미래를 고민하는 이들에게 이 책을 추천한다. 노화 극복이라는 인류의 오랜 꿈에 담긴 딜레마를 흥미로운 추리 속에 풀어 낸 이 책은, 인위적 진화를 시도하는 기술 발전을 깊이 있게 성찰하고 미래 사회에 대한 윤리적 감수성을 키우는 데 도움을 줄 것이다. 
텔로미어 9791170612025
『처절한 무죄』로 제1회 대한민국 콘텐츠공모전 최우수상, 『30년』으로 제1회 갤럭시탭 삼성문학상을 수상하며 주목을 받아온 작가 박성신의 신작 장편 미스터리 『텔로미어』가 ‘북다’에서 출간되었다. 현재 영화화 진행 중인 장편소설 『제3의 남자』로 “치밀한 플롯, 매력적인 캐릭터, 탁월한 밀당 능력. 괴력에 가까운 흡인력이 인상적”(정유정 작가)이라는 평가를 받은 그는, 『텔로미어』로 제10회 교보문고 스토리공모전 우수상을 받으며 다시 한번 균형감


In [46]:
len(json_data)

1052

In [47]:
import json

# Save the data as JSON file with Korean encoding
with open('book_data.json', 'w', encoding='utf-8') as f:
    json.dump(json_data, f, ensure_ascii=False, indent=4)